# HW1: COVID-19 Cases Prediction (Regression)

Kaggle competition: [`ml2021spring-hw1`](https://www.kaggle.com/competitions/ml2021spring-hw1)

Metric: RMSE on `tested_positive`. This notebook is meant to be run on Kaggle with the competition added as a data source (no internet needed).

**Result of this notebook**

| model | public | private |
|---|---|---|
| reference public baseline notebook | 0.88137 | 0.89887 |
| this notebook: single 14-feature MLP | 0.88744 | 0.90020 |
| this notebook: single 17-feature MLP | 0.88129 | 0.89380 |
| previous best (Huber + acceleration) | 0.90267 | 0.88526 |
| **final 5-model ensemble** | **0.87955** | **0.88478** |

Strong baseline is 0.88 on the private LB, so the final ensemble is still about 0.005 short. The last section explains what was tried.


## 1. Data quirks

Three things in this dataset are easy to get wrong:

1. **The target column is also called `tested_positive`.** The CSV has day-1 `tested_positive`, day-2 `tested_positive.1`, and the target `tested_positive.2`. `df['tested_positive']` returns the day-1 *feature*, not the target. Always use `df.iloc[:, -1]`.
2. **Rows are grouped by state**, 67 or 68 consecutive days per state (2700 rows / 40 states). The test set is the following 22-23 days per state (893 rows).
3. **The official `i % 10 == 0` split leaks.** Because rows are consecutive days of the same state, the target of a validation row equals `tested_positive.1` (day-2 feature) of the next row, which is usually in the training split. Random CV therefore looks far better than the leaderboard.


In [ ]:
import os, glob, time, math, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
print('torch', torch.__version__, '| device', device)

tr_path = glob.glob('/kaggle/input/**/covid.train.csv', recursive=True)[0]
tt_path = glob.glob('/kaggle/input/**/covid.test.csv', recursive=True)[0]
tr = pd.read_csv(tr_path)
te = pd.read_csv(tt_path)
print('shapes:', tr.shape, te.shape)
print('last 3 columns:', list(tr.columns[-3:]))

# positional indices: 0=id, 1..40 states, 41..58 day1, 59..76 day2, 77..93 day3, 94=target
y = tr.iloc[:, -1].values.astype(np.float32)
st = tr.iloc[:, 1:41].values.argmax(1)
st_te = te.iloc[:, 1:41].values.argmax(1)
tp1, tp2 = 58, 76
print('target mean/std/min/max: %.3f %.3f %.3f %.3f' % (y.mean(), y.std(), y.min(), y.max()))
print('rows per state:', pd.Series(st).value_counts().unique())


## 2. Features

`F14` is the feature set selected by the public strong-baseline notebook:
`tested_positive` of day 1 and day 2, plus `cli`, `ili`, `hh_cmnty_cli`, `nohh_cmnty_cli` over the three days
(4 symptoms x 3 days + 2 lags = 14).

`F17` adds `worried_finances` for the three days (the private strong-baseline variant).

`F_eng` adds changes and accelerations of the four symptoms, plus the tested-positive change and ratio.


In [ ]:
for df in (tr, te):
    df['tp_delta'] = df.iloc[:, tp2] - df.iloc[:, tp1]
    df['tp_ratio'] = df.iloc[:, tp2] / (df.iloc[:, tp1] + 0.1)
    for name, (a, b, c) in {'cli': (41, 59, 77), 'ili': (42, 60, 78),
                            'hh': (43, 61, 79), 'nohh': (44, 62, 80)}.items():
        df[f'{name}_d21'] = df.iloc[:, b] - df.iloc[:, a]
        df[f'{name}_d32'] = df.iloc[:, c] - df.iloc[:, b]
        df[f'{name}_acc'] = df.iloc[:, c] - 2 * df.iloc[:, b] + df.iloc[:, a]

states = list(range(1, 41))
F14 = [tp1, tp2, 41, 59, 77, 42, 60, 78, 43, 61, 79, 44, 62, 80]
F17 = F14 + [57, 75, 93]                    # + worried_finances x 3
F_eng = F17 + ['cli_d21', 'cli_d32', 'cli_acc', 'ili_d21', 'ili_d32', 'ili_acc',
               'hh_d21', 'hh_d32', 'hh_acc', 'nohh_d21', 'nohh_d32', 'nohh_acc',
               'tp_delta', 'tp_ratio']
print({k: len(v) for k, v in {'F14': F14, 'F17': F17, 'F_eng': F_eng}.items()})


def build(df, feats):
    cols = [df.iloc[:, f].values if isinstance(f, int) else df[f].values for f in feats]
    return np.stack(cols, axis=1).astype(np.float32)


## 3. Leak-free validation

Validate on the **last 20% of days of every state**. This mimics the test set, which is the future days of every state. The official random split gives an RMSE around 0.1 and is useless for model selection.

The persistence baseline `y = tested_positive.1` (day 2) already reaches about 1.18 RMSE on this split, so most of the signal is in the two tested-positive lags.


In [ ]:
def perstate_split(frac=0.2):
    tr_idx, va_idx = [], []
    for s in range(40):
        ids = np.where(st == s)[0]
        cut = int(len(ids) * (1 - frac))
        tr_idx += ids[:cut].tolist()
        va_idx += ids[cut:].tolist()
    return np.array(tr_idx), np.array(va_idx)

trn, va = perstate_split(0.2)
print('persistence RMSE (time split): %.4f' %
      np.sqrt(np.mean((tr.iloc[va, tp2].values - y[va]) ** 2)))


def ridge_fit(Xtr, ytr, Xte, lam=0.1):
    '''Closed-form ridge with intercept, standardized features.'''
    mu = Xtr.mean(0); sd = Xtr.std(0); sd = np.where(sd < 1e-9, 1.0, sd)
    A = np.c_[np.ones(len(Xtr)), (Xtr - mu) / sd]
    B = np.c_[np.ones(len(Xte)), (Xte - mu) / sd]
    G = A.T @ A
    G[1:, 1:] += lam * np.eye(Xtr.shape[1])
    w = np.linalg.solve(G, A.T @ ytr)
    return B @ w

X17 = build(tr, F17).astype(np.float64)
for lam in [0.1, 1.0, 10.0]:
    p = ridge_fit(X17[trn], y[trn], X17[va], lam)
    print('ridge F17 lam=%-4g time20 RMSE %.4f' % (lam, np.sqrt(np.mean((p - y[va]) ** 2))))


## 4. Models

The ensemble below uses five components:

* **MLP** on `F_eng`: one hidden layer of 32, BatchNorm, Dropout 0.2, LeakyReLU, Adam (lr 1e-3), L2 7.5e-4, Huber loss (delta 2). Early stopping on the train loss with patience 1200, which stops around epoch 500-1000. Training longer overfits badly: a fixed 6000-epoch run scores about 0.94 on the LB.
* **Ridge** on `F17` (lambda 0.1).
* **Ridge** on `F17 + 40 state one-hots` (lambda 0.1).
* **ExtraTrees** (500 trees, `min_samples_leaf=2`, `max_features=0.7`) on `F17`.
* **RandomForest** (400 trees, `min_samples_leaf=2`, `max_features=0.6`) on `F17`.

The single MLP is the strongest component (~0.894 private); ridge and trees are weaker alone but add useful diversity.
The final weights were chosen as `0.30 * MLP + 0.30 * ridge17 + 0.20 * ridge_state + 0.05 * ExtraTrees + 0.15 * RandomForest`.


In [ ]:
class MLP(nn.Module):
    def __init__(self, d, hidden=32, drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, hidden), nn.BatchNorm1d(hidden), nn.Dropout(drop),
            nn.LeakyReLU(0.01), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(feats, hidden=32, lr=1e-3, l2=7.5e-4, batch=200, delta=2.0,
              max_epochs=6000, patience=1200, seed=42069):
    torch.manual_seed(seed); np.random.seed(seed)
    X = build(tr, feats); Xte = build(te, feats)
    xt = torch.tensor(X); yt = torch.tensor(y); xv = torch.tensor(Xte)
    model = MLP(len(feats), hidden).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, bad, best_ep = 1e18, None, 0, 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(xt), device=device)
        for i in range(0, len(xt) - 1, batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            loss = nn.functional.huber_loss(model(xt[idx]), yt[idx], delta=delta)
            loss = loss + l2 * sum(p.pow(2).sum() for p in model.parameters())
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            tl = nn.functional.huber_loss(model(xt), yt, delta=delta).item()
        if tl < best - 1e-6:
            best, best_ep, bad = tl, ep + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pred = model(xv).cpu().numpy()
    print('MLP stopped at epoch %d, train Huber %.4f' % (best_ep, best))
    return pred


pred = {}
pred['mlp'] = train_mlp(F_eng)

pred['ridge17'] = ridge_fit(X17, y, build(te, F17).astype(np.float64), 0.1)

XS = build(tr, states + F17).astype(np.float64)
XSte = build(te, states + F17).astype(np.float64)
pred['ridge_state'] = ridge_fit(XS, y, XSte, 0.1)

et = ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, max_features=0.7,
                         random_state=0, n_jobs=-1).fit(X17, y)
pred['et'] = et.predict(build(te, F17).astype(np.float64))

rf = RandomForestRegressor(n_estimators=400, min_samples_leaf=2, max_features=0.6,
                           random_state=0, n_jobs=-1).fit(X17, y)
pred['rf'] = rf.predict(build(te, F17).astype(np.float64))


## 5. Blend and submission

In [ ]:
final = (0.30 * pred['mlp'] + 0.30 * pred['ridge17'] + 0.20 * pred['ridge_state']
         + 0.05 * pred['et'] + 0.15 * pred['rf'])

sub = pd.DataFrame({'id': te['id'].values, 'tested_positive': final})
sub.to_csv('submission.csv', index=False)
print('saved submission.csv', sub.shape)
print(sub['tested_positive'].describe())

for k, v in pred.items():
    print('%-12s mean %.3f std %.3f min %.2f max %.2f' %
          (k, v.mean(), v.std(), v.min(), v.max()))


## 6. What was tried, and why it stops here

* Reproduced the 14-feature public baseline: private **0.9002** (the public notebook's own output scores 0.8989).
* Reproduced the 17-feature private-baseline recipe with train-loss early stopping: private **0.8938**.
* Huber loss on the 14/17/31-feature sets: 0.893-0.900 private.
* Extra features (all 93, state one-hots, national daily means, state-relative values, acceleration terms): no consistent gain on the time split.
* Residual models (`y - tp2`, `y - trend`): worse on the LB (0.91-0.93).
* Seed ensembles, snapshot epochs, ridge/tree blends: the 5-model blend above is the best at **0.88478 private / 0.87955 public**.

The remaining gap to 0.88 is small but real. The likely next steps are a proper stacked ensemble with out-of-fold meta-features, or a model that handles the future time shift better than a global MLP.
